## 1. Question, definitions, and pilot scope

Both analyses matter equally: freeze a reference factor and examine its meaning; freeze a reference rule and examine its representation. Keep rule–factor agreement, factor–prediction influence and prediction–outcome reliability separate. Annual same-year mortality is the existing classifier target. The forecast target is next-year subgroup Brier-loss change. This configuration scaffold has no scientific results yet.


## 2. Configuration and repository integration

Visible approved presets below; profiles are `v0` (tiny synthetic CPU fixtures), `pilot` (full compute, then analysis) and `analysis` (no compute; reports on completed units of the same configuration hash). Paths come from explicit host inputs. No dataset or cached predictor is read by configuration fixtures. There is no time budget: compute ends on completion, the disk guard or a user stop, and resumes from completed units with the same configuration hash. The configuration hash covers every scientific setting and excludes profile, device and paths. Configuration fixtures are tested independently of scientific stages. Authoring and all execution occur on Fiji; laptop holds review copies only.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import sys
from jupyter_client.kernelspec import KernelSpecManager

# Import the snapshot helper, never a different working checkout's helper.
SOURCE_ROOT = Path.cwd().absolute()
sys.path.insert(0, str(SOURCE_ROOT))
import concept_subgroup_drift_pilot as pilot
assert Path(pilot.__file__).absolute() == SOURCE_ROOT / "concept_subgroup_drift_pilot.py", "Wrong snapshot helper"
KERNEL = KernelSpecManager().get_kernel_spec("sae-tcav-fiji")
KERNEL_ROOT = Path(os.environ["RPI_RUN_DIR"]) / "kernel" / "share" / "jupyter" / "kernels"
assert Path(KERNEL.resource_dir).resolve().is_relative_to(KERNEL_ROOT.resolve()), "Kernel is not run-local"
KERNEL_SPEC = KERNEL.to_dict()
KERNEL_IDENTITY = pilot.verify_notebook_environment(
    os.environ.get("PILOT_EXPECTED_PYTHON"), KERNEL_SPEC)
print("Fiji executable/kernel identity:", KERNEL_IDENTITY)
print("Snapshot helper SHA256:", hashlib.sha256(Path(pilot.__file__).read_bytes()).hexdigest())


In [ ]:
PRESET_SPEC = {'shared': {'split_seed': 42,
            'reference_year_candidates': [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015],
            't0_only_reference_years': [2015],
            'sae_seeds': [42, 135, 215],
            'epochs': 100,
            'threshold_sweep': [0.1, 0.2, 0.3, 0.4, 0.5],
            'rule': {'objective': 'f2',
                     'min_precision': 0.5,
                     'min_lift': 1.5,
                     'max_rules': 5,
                     'max_rule_length': 10,
                     'min_marginal_recall': 0.02,
                     'rule_source': 'semantic',
                     'recurrence': {'max_candidates_per_bootstrap': 20,
                                    'family_similarity_threshold': 0.7,
                                    'min_family_recurrence': 0.5}},
            'candidate_cap_fraction': 0.1,
            'matching_roles': ['rule_discovery', 'cav_fit'],
            'chance_permutations': 20,
            'cav_seeds': [42, 43, 44],
            'primary_cav_seed': 42,
            'tcav_cutoffs': [0.1, 0.9],
            'bootstrap_replicates': 100,
            'confidence_level': 0.95,
            'classification_threshold': 0.5,
            'label_availability': 'end_of_year',
            'encoding_batch_size': 256,
            'amp': 'off',
            'tf32': False,
            'disk_guard_gib': 15,
            'joblib_workers': 32,
            'inference_batch_rows': 4096,
            'sae': {'model_type': 'ReLU',
                    'scaling_factor': 1.5,
                    'alpha': 0.1,
                    'learning_rate': 0.001,
                    'weight_decay': 0.0},
            'data_path': None,
            'population_sha256': None,
            'cache_dir': None,
            'output_dir': None},
 'v0': {'device': 'cpu',
        'epochs': 2,
        'reference_year_candidates': [2007, 2008, 2009],
        't0_only_reference_years': [2009],
        'chance_permutations': 5,
        'bootstrap_replicates': 10,
        'joblib_workers': 2,
        'inference_batch_rows': 64},
 'pilot': {'device': 'cuda',
           'population_sha256': 'c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7'},
 'analysis': {'device': 'cpu',
              'population_sha256': 'c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7'}}

In [ ]:
PROFILE = os.environ.get("PILOT_PROFILE", "v0")
CONFIG = pilot.resolve_config(PRESET_SPEC, PROFILE, pilot.configuration_overrides(os.environ))
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print("Profile:", PROFILE, "; device:", CONFIG["device"], "; configuration hash:", CONFIG["config_hash"])
print("Reference years:", CONFIG["reference_year_candidates"], "; T0-only (no follow-up):", CONFIG["t0_only_reference_years"])
print("Split seed:", CONFIG["split_seed"], "; matching roles:", CONFIG["matching_roles"])
print("SAE seeds:", CONFIG["sae_seeds"], "; epochs:", CONFIG["epochs"])
print("Threshold sweep A:", CONFIG["threshold_sweep"], "; rule settings:", CONFIG["rule"])
print("Candidate cap at 288 latents:", pilot.candidate_cap(288, CONFIG["candidate_cap_fraction"]))
print("Chance permutations:", CONFIG["chance_permutations"], "; CAV seeds:", CONFIG["cav_seeds"], "; bootstrap:", CONFIG["bootstrap_replicates"])
print("Disk guard (GiB):", CONFIG["disk_guard_gib"], "; joblib workers:", CONFIG["joblib_workers"], "; cache dir:", CONFIG["cache_dir"])
print("Compute has no time limit; it ends on completion, disk guard or user stop. Runtime estimates: unknown.")
if PROFILE == "v0":
    pilot.run_configuration_checks()
DEVICE = pilot.resolve_device(CONFIG["device"])
CONFIG["device"] = DEVICE
OUTPUT = Path(CONFIG["output_dir"] or os.environ["RPI_OUT_DIR"])
assert OUTPUT.is_absolute(), "Output path must be explicit and absolute"
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = Path(CONFIG["cache_dir"] or OUTPUT / "unit-cache")
if PROFILE != "v0":
    assert CONFIG["cache_dir"] and CACHE_ROOT.is_absolute(), "Scientific profiles require explicit absolute PILOT_CACHE_DIR"
if PROFILE == "analysis":
    assert (CACHE_ROOT / CONFIG["config_hash"] / "config.json").is_file(), "Analysis cache absent; no completed run to analyze"
SOURCE_IDENTITIES = {name: hashlib.sha256((SOURCE_ROOT / name).read_bytes()).hexdigest()
                     for name in ("concept_subgroup_drift_pilot.py", "concept_subgroup_drift_pilot.ipynb")}
SOURCE_HASH = hashlib.sha256(json.dumps(SOURCE_IDENTITIES, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
CACHE = pilot.UnitCache(CACHE_ROOT, CONFIG, SOURCE_HASH, read_only=(PROFILE == "analysis"))
import random
import numpy as np
import torch
random.seed(CONFIG["split_seed"])
np.random.seed(CONFIG["split_seed"])
torch.manual_seed(CONFIG["split_seed"])
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(CONFIG["split_seed"])
import importlib.metadata
import socket
import subprocess
import time
from datetime import datetime, timezone
START_CLOCK = time.monotonic()
LAUNCH_PATH = Path(os.environ["RPI_RUN_DIR"]) / "run.json"
GIT_ROOT = Path(os.environ.get("RPI_GIT_ROOT", SOURCE_ROOT))
def git_read(*arguments):
    return subprocess.check_output(["git", *arguments], cwd=GIT_ROOT, text=True).strip()
PACKAGES = {}
for package in ("torch", "numpy", "pandas", "scikit-learn", "tabpfn", "joblib", "matplotlib", "nbconvert"):
    try: PACKAGES[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: PACKAGES[package] = None
ACCELERATOR = {"requested_device": PRESET_SPEC[PROFILE]["device"], "resolved_device": DEVICE,
               "torch_version": torch.__version__, "cuda_version": torch.version.cuda,
               "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"), "amp": CONFIG["amp"], "tf32": CONFIG["tf32"]}
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(torch.cuda.current_device())
    ACCELERATOR.update(device_name=properties.name, total_memory_bytes=properties.total_memory,
                       compute_capability=[properties.major, properties.minor])
MANIFEST = {"schema_version": "1", "run_id": Path(os.environ["RPI_RUN_DIR"]).name,
            "status": "running", "profile": PROFILE, "command": sys.orig_argv, "host": socket.gethostname(),
            "started_utc": datetime.now(timezone.utc).isoformat(), "ended_utc": None,
            "git": {"sha": git_read("rev-parse", "HEAD"), "branch": git_read("branch", "--show-current") or None,
                    "dirty": bool(git_read("status", "--porcelain"))},
            "source_file_hashes": SOURCE_IDENTITIES, "source_hash": SOURCE_HASH,
            "config": {"hash": CONFIG["config_hash"], "resolved": CONFIG}, "seed": CONFIG["split_seed"],
            "deterministic": bool(torch.are_deterministic_algorithms_enabled()),
            "accelerator": ACCELERATOR, "environment": {"python": sys.version, "executable": sys.executable, "packages": PACKAGES},
            "launcher": json.loads(LAUNCH_PATH.read_text()) if LAUNCH_PATH.exists() else None,
            "data": [], "prerequisites": [], "error": None,
            "cache_dir": str(CACHE.directory), "outputs": [],
            "scheduling": "P ascending; S seed-major; D year-major; CPU rule payload overlaps main-thread yearly SAE units; cache writes main-thread only"}
STOP_REASON = None
pilot._atomic_json(OUTPUT / "resolved-config.json", CONFIG)
def save_progress():
    records = [CACHE._latest[key] for key in pilot.unit_order(CONFIG) if key in CACHE._latest]
    pilot._atomic_json(OUTPUT / "stages.json", {"stages": records, "stop_reason": STOP_REASON})
    pilot._atomic_json(OUTPUT / "run-manifest.json", MANIFEST)
    if CACHE._manifest.exists():
        (OUTPUT / "units.jsonl").write_bytes(CACHE._manifest.read_bytes())
def execute_unit(key, function):
    global STOP_REASON
    if STOP_REASON:
        return None
    resumed = CACHE.is_complete(key)
    try:
        record = function()
    except pilot.DiskGuardStop:
        STOP_REASON = "stopped:disk_guard"
        print(STOP_REASON, key)
        save_progress()
        return None
    save_progress()
    complete = sum(CACHE.is_complete(item) for item in CACHE.graph)
    timings = [record["seconds"] for record in CACHE._latest.values() if record.get("seconds") is not None]
    eta = sum(timings) / len(timings) * (len(CACHE.graph) - complete) if timings else None
    print(key, record["status"], record["reason"], "resume", resumed, "seconds", record["seconds"],
          "peak GPU bytes", record["peak_gpu_bytes"], "complete", complete, "/", len(CACHE.graph), "rough ETA seconds", eta, flush=True)
    return record
save_progress()

## 3. Data, patient splits, and predictive system

Patient roles reuse the reference-only split (seed 42), subdividing rule-selection/CAV patients equally into CAV fitting and independent selection. The internal evaluation role maps to the original t0_evaluation role. Entrants retain a deterministic SHA-256 assignment. The annual target is DEATH in the same patient-year, not next-year mortality. Feature counts exclude DEATH. Observation presence is audited against selected features plus DEATH; additional rows require an eligibility decision. Protected outcome-dependent audits wait for final evaluation.

The supported predictor is installed drift-resilient single_fast with fit_at_predict_time=False, one cached configuration, and fixed reference context/domain. Capture probabilities and decoder-input embeddings in one call. Replay the actual executed nonlinear decoder, class truncation, frozen log-temperature, class permutation and original aggregation/reweighting. Rounding/clipping precision mode is unsupported. Synthetic fixtures check the mortality probability objective and finite differences. The fixed approved central-difference step is 0.01; derivative rtol=0.001 and atol=0.00001 stay unchanged. Report probabilities on both perturbations, derivatives, signal, errors and elementwise tolerance. The earlier 0.001-step failure remains recorded; no adaptive step or favorable row/direction search. Real-data CUDA parity remains a prerequisite.

The population is the retained prepared test partition, one row per patient-year (21,900 rows, 500 features, years 2007-2015 on Fiji). Its file bytes are verified against the configured sha256 before anything is unpickled; a mismatch is refused. The v0 profile builds a tiny synthetic population in-process instead. Roles are assigned per reference year r from year-r patients and year-r outcomes only: returning reference patients keep their role in every later year, and patients absent from year r are entrants with a stable hashed fitting/selection/evaluation role (experiment identity r). Rows of years before r carry no role. The fitting-side support table lists rows, patients, deaths, survivors and post-death rows by role and year; deaths of later-year held-out roles are withheld until the evaluation units.

The P(r) unit fits the frozen predictor once on year-r tabpfn_context rows, runs the identity, order, batch, parity and finite-difference gates on the run device, then captures probabilities and raw embeddings for every row of years r..last in one batched pass (inference-only out-of-memory backoff, recorded). A failed gate marks that reference year failed and its dependents unavailable; other years continue.


In [ ]:
# Pilot prerequisites precede every compute unit; analysis performs no model work.
import gc
import xml.etree.ElementTree as ET
def prerequisite(name, function):
    entry = {"name": name, "status": "running", "started_utc": datetime.now(timezone.utc).isoformat()}
    MANIFEST["prerequisites"].append(entry)
    save_progress()
    try:
        function()
    except BaseException as error:
        entry.update(status="failed", reason=str(error))
        MANIFEST.update(status="failed", error={"type": type(error).__name__, "message": str(error)},
                        ended_utc=datetime.now(timezone.utc).isoformat())
        save_progress()
        raise
    else:
        entry["status"] = "passed"
    finally:
        entry["ended_utc"] = datetime.now(timezone.utc).isoformat()
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
        save_progress()
def gpu_embedding_gate():
    assert DEVICE == "cuda", "Pilot requires CUDA prerequisite execution"
    assert os.environ.get("TEMPORAL_REUSE_SMOKE_MANIFEST"), "TEMPORAL_REUSE_SMOKE_MANIFEST required for CUDA replay gate"
    test = SOURCE_ROOT / "tests" / "test_temporal_embedding_reuse_gpu.py"
    assert test.is_file(), "Immutable snapshot missing GPU prerequisite test"
    result = OUTPUT / "gpu-prerequisite.xml"
    subprocess.run([sys.executable, "-B", "-m", "pytest", "-q", "-rs", "-p", "no:cacheprovider",
                    str(test), "--junitxml", str(result)], check=True)
    required = [entry for entry in ET.parse(result).iter("testcase")
                if entry.attrib.get("name") == "test_cuda_skipped_training_pass_matches_the_legacy_path"]
    assert len(required) == 1 and not list(required[0]), "Required CUDA replay skipped or failed"
if PROFILE == "pilot":
    prerequisite("cuda_embedding_replay", gpu_embedding_gate)
    prerequisite("predictor_sdpa_compatibility", pilot.run_predictor_sdpa_checks)
    for check in (pilot.run_reference_sae_unit_checks, pilot.run_family_checks, pilot.run_cav_selection_checks,
                  pilot.run_idea2_checks, pilot.run_group_intervention_checks):
        prerequisite(check.__name__ + "_cuda_tolerances", check)
if PROFILE != "analysis":
    if PROFILE == "v0":
        POPULATION = pilot.synthetic_population(years=CONFIG["reference_year_candidates"])
    else:
        assert CONFIG["data_path"] and Path(CONFIG["data_path"]).is_absolute(), "Pilot requires explicit absolute PILOT_DATA_PATH"
        POPULATION = pilot.load_pilot_population(CONFIG["data_path"], CONFIG["population_sha256"])
    MANIFEST["data"] = [POPULATION["fingerprint"]]
    ROLES_BY_YEAR = {r: pilot.reference_year_roles(POPULATION, r, CONFIG["split_seed"])
                     for r in CONFIG["reference_year_candidates"]}
    SUPPORT = [dict(row) for r, roles in ROLES_BY_YEAR.items()
               for row in pilot.role_support_table(POPULATION, roles)]
    pilot.write_role_support_csv(SUPPORT, OUTPUT / "tables" / "role-support.csv")
    for r, roles in ROLES_BY_YEAR.items():
        execute_unit(f"P/{r}", lambda r=r, roles=roles: CACHE.run(f"P/{r}",
                     lambda: pilot.predictor_unit(POPULATION, r, roles, CONFIG, DEVICE),
                     projected_bytes=pilot.projected_predictor_bytes(POPULATION, r)))
else:
    print("Analysis reads committed cache units only; patient population unopened.")

## 4. Reference SAE families, factors, rules, and CAVs

Reference fitting supplies one frozen scaler and three seeded SAEs per reference year. Factor families are built without an anchor seed. The year-r rule_discovery and cav_fit rows (the matching rows; embeddings only, no outcomes, and never the SAE discovery rows) are encoded with every seed's SAE. A latent that is exactly zero on every matching row is dead and excluded; a constant nonzero latent has no defined correlation and is excluded and counted separately. For every seed pair, the Pearson correlation of continuous activations (zeros included) is computed for every factor pair, in float64 by a z-scored matrix product on the run device. Factors i and j are linked when each is the other's highest-correlation factor across the two seeds (mutual nearest neighbours): there is no similarity threshold, and ties go to the lowest factor ID and are recorded. Connected components of the link graph are classified as full (one factor per seed, all three pairs linked), partial (exactly two factors of different seeds), chain (one factor per seed without every pair linked; reported, never a candidate), inconsistent (two factors of one seed, a split or merge; reported, never forced) or singleton. A family ID lists its sorted (seed, factor) members; its strength is the minimum and median cross-seed correlation inside it, reported and never gated; its medoid, the member with the highest mean correlation to the others (ties: lowest seed, factor), represents it. A chance check repeats the procedure with every seed's row order independently permuted (configured permutations, seeded) and reports the observed counts against the permutation mean and maximum; it validates and never selects. Candidates are full families ranked by medoid positive-activation count on the matching rows (ties by family ID), capped at ceil(0.10 x latents); the D(r) families unit caches compact family, member and candidate tables.

Reference rules follow the original semantic procedure for every candidate family medoid, using only the year-r rule_discovery rows: the medoid activations come from the families unit by row identity (alignment asserted) and the cav_fit rows are never read. For each A in the configured sweep (0.1 to 0.5) a positive-only cutoff is fitted on the rule_discovery activations (the top A of the strictly positive activations, at least two positives); the binary target is activation > cutoff, and a target without enough positives marks that A invalid with its reason. The rule_discovery patients are split once per family into deterministic fit and tune halves (patient level, stratified by medoid positive activation, seeded from the split seed, reference year and family ID), shared by every A. The stable rule backend (30 bootstraps x 50 trees, depth 15, seeded per family and A) proposes candidates on the fit half; as in the original pipeline, candidate occurrences are capped per bootstrap (max_candidates_per_bootstrap), clustered into rule families on the tune half (family_similarity_threshold) and only families recurring in at least min_family_recurrence of the bootstraps reach the constrained rule-set selection (the three values are visible in the rule presets); it runs on the tune half with objective F2, minimum precision 0.5, minimum lift 1.5, at most 5 rules of at most 10 conditions and minimum marginal recall 0.02. Every A keeps a diagnostic row (feasibility, reason, cutoff, positives per half, tune-half F2, precision, recall and lift, rule and condition counts, rule identity). The winner among feasible A follows the original ordering (F2, then precision, lift, fewer rules, fewer conditions, smaller A); its cutoff becomes the frozen activation threshold and its rule set the frozen rule. A family with no feasible A is dropped as no_feasible_semantic_rule and recorded; the candidate list is never refilled, so later stages see only the surviving families. Family-by-A jobs run in a joblib process pool (configured workers) with deterministic per-job seeds, so results do not depend on the worker count. rule_source is a parameter; only semantic is enabled in this pilot.

The D(r) CAV and selection unit works on the surviving (frozen-rule) families only. Per family it fits CAV seeds 42/43/44 in raw predictor coordinates on the year-r cav_fit rows alone: positives are the rows where the frozen rule fires (evaluated in float64, like the backend that fitted it) and negatives are rule-negative rows whose medoid activation equals the minimum finite activation of those cav_fit rows (positive_fraction .10 puts the negative-pool quantile at zero; the pool cutoff is recorded with each CAV). Each seed trains on a balanced random subsample, so seeds differ in the sampled rows. Prerequisites are computational only: at least two rule positives, two negatives, two distinct patients per class, finite embeddings and a finite nonzero direction. A repetition that fails any of them is recorded as unavailable with its reason and is never replaced. Selection uses the year-r concept_selection rows alone and the gradients of the final death probability with respect to the raw embeddings cached by the predictor unit (the predictor is never rerun). Idea 1 measures the raw seed-42 TCAV (the share of rows whose gradient has a positive projection on the CAV) inside subgroup A, the rows whose medoid activation is strictly above the frozen winner cutoff; idea 2 measures it separately inside subgroup R, the rows where the frozen rule fires. A family is eligible for an idea when that TCAV is <=0.1 or >=0.9 (both edges inclusive); the two ideas are independent, an empty subgroup or an unavailable seed-42 CAV makes that idea unavailable with its reason, and seeds 43/44 are recorded as diagnostics only. The unit records, per family, the CAV status per seed, the subgroup inventory (A, R, their intersection and complement counts), the scores and the eligibility or reason per idea, and freezes the target, rule, CAV vectors and their identity so that later stages refuse a changed object. Mortality, rule quality, significance and performance never gate eligibility. The S(seed, r) unit trains one reference SAE per SAE seed (full batch, GPU-resident input, configured epochs) on year-r sae_discovery embeddings taken by role from the cached P(r) output. The reference scaler is fitted deterministically on exactly those rows in every unit, recorded with its identity, and must be identical across seeds. SAE state, scaler, latent count, final loss and training telemetry are cached; a nonfinite loss or output fails the unit and makes its dependents unavailable. The P(r) unit also caches the death-probability gradients of every eligible row and a serializable copy of the executed predictor head (layers, tensors, class permutation, log-temperature and aggregation settings) with an identity hash; load_continuation rebuilds it without refitting the predictor and refuses it unless it reproduces the cached probabilities (rtol 1e-4, atol 1e-5) and gradients (rtol 1e-3, atol 1e-5) on the cached gate rows.


In [ ]:
# Reference SAE stage is seed-major. Downstream stage finishes one reference year at a time.
def compute_downstream(r, roles):
    global STOP_REASON
    matching = pilot.matching_population_view(POPULATION)
    rules = pilot.rule_population_view(POPULATION)
    performance = pilot.performance_population_view(POPULATION)
    execute_unit(f"D/{r}/families", lambda: pilot.run_family_unit(CACHE, matching, roles, CONFIG, DEVICE))
    rule_key = f"D/{r}/rules"
    yearly_jobs = [(f"D/{r}/idea2_yearly_sae/{seed}/{t}", seed, t)
                   for t in pilot.idea2_follow_up_years(CONFIG, r) for seed in CONFIG["sae_seeds"]]
    def train_yearly(key, seed, t):
        return execute_unit(key, lambda: pilot.run_idea2_yearly_sae_unit(CACHE, matching, roles, CONFIG, DEVICE, seed, t))
    if not STOP_REASON and not CACHE.is_complete(rule_key) and CACHE.is_complete(f"D/{r}/families"):
        families = CACHE.load(f"D/{r}/families")
        # CPU worker returns a payload. All cache writes and CUDA calls stay on this notebook thread.
        def rule_payload():
            payload, _ = pilot.overlap_cpu_gpu(
                lambda: pilot.rule_unit(rules, r, roles, families, CONFIG, workers=CONFIG["joblib_workers"]),
                [lambda key=key, seed=seed, t=t: train_yearly(key, seed, t) for key, seed, t in yearly_jobs])
            return payload
        execute_unit(rule_key, lambda: CACHE.run(rule_key, rule_payload, projected_bytes=pilot.projected_rule_bytes(CONFIG)))
    else:
        execute_unit(rule_key, lambda: pilot.run_rule_unit(CACHE, rules, roles, CONFIG))
    if STOP_REASON: return
    execute_unit(f"D/{r}/cavs_selection", lambda: pilot.run_cavs_selection_unit(CACHE, rules, roles, CONFIG, DEVICE))
    execute_unit(f"D/{r}/reference_performance", lambda: pilot.run_reference_performance_unit(CACHE, performance, roles, CONFIG, DEVICE))
    for t in pilot.idea1_follow_up_years(CONFIG, r):
        execute_unit(f"D/{r}/idea1/{t}", lambda t=t: pilot.run_idea1_unit(CACHE, performance, roles, CONFIG, DEVICE, t))
        execute_unit(f"D/{r}/idea2_frozen/{t}", lambda t=t: pilot.run_idea2_frozen_unit(CACHE, rules, roles, CONFIG, DEVICE, t))
        for seed in CONFIG["sae_seeds"]:
            train_yearly(f"D/{r}/idea2_yearly_sae/{seed}/{t}", seed, t)
        execute_unit(f"D/{r}/idea2_yearly_families/{t}", lambda t=t: pilot.run_idea2_yearly_families_unit(CACHE, matching, roles, CONFIG, DEVICE, t))
        execute_unit(f"D/{r}/groups_interventions/{t}", lambda t=t: pilot.run_groups_interventions_unit(CACHE, performance, roles, CONFIG, DEVICE, t))
        if STOP_REASON: return
if PROFILE != "analysis":
    for seed in CONFIG["sae_seeds"]:
        for r, roles in ROLES_BY_YEAR.items():
            execute_unit(f"S/{seed}/{r}", lambda seed=seed, roles=roles: pilot.run_sae_unit(CACHE, POPULATION, roles, CONFIG, seed, DEVICE))
    for r, roles in ROLES_BY_YEAR.items():
        compute_downstream(r, roles)
        if STOP_REASON: break
REPORT = pilot.report_tables(CACHE, CONFIG)
import pandas as pd
from IPython.display import display
INCLUDED_UNITS = [row["unit"] for row in REPORT["completeness"] if row["cache_status"] == "complete"]
print("Included committed units:", INCLUDED_UNITS)
print("Missing/unavailable/failed units:", [row for row in REPORT["completeness"] if row["status"] != "complete"])
def show_tables(*names):
    for name in names:
        print(name, "; included reference years:", sorted({row.get("reference_year") for row in REPORT[name] if row.get("reference_year") is not None}))
        display(pd.DataFrame(REPORT[name]))
show_tables("families", "members", "permutation", "sweep", "rules", "selection")

## 5. Reference subgroup performance

For each subgroup, q is the mean mortality label on separate reference fitting patients. Evaluate Brier_model=mean((p-y)^2), Brier_baseline=mean((q-y)^2), and skill=1-Brier_model/Brier_baseline on identical held-out rows. Positive skill remains continuous evidence of improvement over this baseline. Empty fitting groups and zero baseline loss yield null skill with reasons; single-class Brier remains valid.

Report fixed-threshold0.5 confusion metrics separately by denominator, AUROC only with both classes, and descriptive calibration using five fixed probability bins. Use 100 paired patient bootstrap repetitions and 95% percentile intervals conditional on frozen fitted objects/q. Repeated records from one patient share bootstrap multiplicity; seeds supply explanation variability, not additional patients. Primary mortality evaluation remains unopened here.

The D(r) reference performance unit evaluates, for every family and idea that was eligible at selection, the year-r evaluation (t0_evaluation) rows inside the frozen subgroup: A (medoid activation strictly above the frozen cutoff) for idea 1 and R (the frozen rule) for idea 2. The probability is the cached final death probability of the frozen predictor. The baseline q is the mean label of the year-r tabpfn_context, sae_discovery, rule_discovery and cav_fit rows that fall in the same frozen subgroup; selection and evaluation rows never enter it. Each row reports counts, deaths, q, Brier, baseline Brier, skill with its bootstrap interval, mean risk against prevalence (calibration), F1, sensitivity and specificity at 0.5, with null values and reasons where the subgroup is empty or has one outcome class. The bootstrap resamples patients (configured replicates, 95% percentile intervals) with a seed derived from the reference year, family and idea, conditional on the frozen objects and q.


In [ ]:
show_tables('performance')

## 6. Idea 1: fixed-factor trajectories and changing rules

For every reference year r and every follow-up year t > r (t = r is covered by the reference units), the D(r)/idea1(t) unit follows each family that was Idea-1-eligible at selection. A_t is defined by the frozen medoid SAE (seed, factor), the frozen reference scaler and the frozen winner threshold: the medoid activation of the cached year-t embeddings (the predictor is not rerun) strictly above the frozen cutoff. A is never redefined by a yearly object.

Evaluation uses the held-out year-t evaluation patients only, returning reference patients and stable entrants counted separately. The frozen reference rule R is evaluated on the same rows and compared with A_t. Precision=intersection/rule count; recall=intersection/A count; F2=5TP/(5TP+4FN+FP); Jaccard=intersection/union; coverage=mask count/eligible cohort count. Zero denominators yield null values with reasons.

The yearly descriptive rule is fitted on the year-t permanent fitting patients present that year (the reference fitting roles plus fitting entrants) against the frozen-threshold target. Candidate generation is the reference one (stable backend, per-bootstrap cap, recurrence-family filter, deterministic patient-stratified 50/50 inner split); the selection has no quality filters (min_precision=0, min_lift=0, min_marginal_recall=0) and keeps the reference objective, rule count and rule length. Evaluation and selection patients never enter the fit. A year without enough fitting support is unavailable with its reason; there is no retry and no reuse of another year's rule. The unit describes each yearly rule against the reference rule: identity, conditions and features added or removed, and the Jaccard of the two row masks on the held-out rows. Descriptive rules never redefine A or replace the reference objects.

Report activation prevalence |A_t|/held-out records and its ratio to the reference-year value, and the mean activation among strictly positive activations, separately. Zero reference denominators are unavailable. Brier and skill inside A_t and inside R use the baseline q fixed on the reference fitting patients (never refitted yearly), paired patient bootstrap with the configured replicates. Frozen CAV repetitions report raw TCAV and signed/absolute sensitivity from the cached year-t gradients, inside A_t and over the whole held-out cohort. Follow frozen eligible factor identities even if future TCAV weakens. Unavailable cells keep their reason; nothing is zero-filled.


In [ ]:
show_tables('idea1')

## 7. Idea 2: fixed-rule trajectories using the frozen SAE

For every reference year r and every follow-up year t > r (t = r is covered by the reference units), the D(r)/idea2_frozen(t) unit follows each family that was Idea-2-eligible at selection. R[k,t] always uses the frozen reference rule: it is evaluated on the held-out year-t evaluation patients (returning reference patients and stable entrants counted separately) and on the reference-year evaluation patients. Only embeddings and features are read; no outcome enters this stage.

Inside R the unit reports, per member of the full reference family (the medoid and every non-medoid member, each under its own frozen SAE seed and the frozen reference scaler): prevalence = share of R rows with strictly positive activation, magnitude = mean activation over all R rows and over positive rows only, each beside the reference-year value and their ratio (null when the reference value is zero). Medoid retention is the share of R rows whose medoid activation is strictly above the frozen threshold cutoff, beside the reference-year share and their ratio. A year without evaluation rows is unavailable with its reason; nothing is zero-filled. Reliability inside R is reported by the Idea 1 units against the frozen baseline.


In [ ]:
show_tables('idea2_frozen')

## 8. Idea 2: yearly retraining and cross-year matching

For every follow-up year t > r, three yearly SAEs (seeds 42/135/215) are trained on the year-t permanent fitting patients (the reference fitting roles plus fitting entrants) under the frozen reference scaler and in the frozen predictor coordinates; no yearly rescaling and no outcomes. For t = r the yearly view reuses the reference SAEs and families and is labelled as reuse; it has no unit.

Yearly families are built with the same anchor-free procedure as the reference families (identical-row Pearson, mutual nearest neighbours, no threshold) on the eligible held-out year-t evaluation rows only, embeddings only. The reference families are encoded on exactly the same ordered rows and the pairing asserts identical ordered row IDs and dimensions. Family-level similarity is the mean Pearson correlation over all reference-member by yearly-member pairs. Links are family-level mutual nearest neighbours between the reference full families and the yearly full families (ties to the lowest family ID, no forced assignment). Per reference family the unit reports retained yes/no, similarity, the runner-up gap to the second most similar yearly full family, and a state: retained, not_mutual_full, partial, chain, inconsistent or unmatched (for example an inactive reference member on the year-t rows). The 20-permutation chance check permutes every yearly seed's row order independently and reports retained links and their similarity under chance beside the observed values; it is validation only. Seeds are explanation variability, not independent patients.


In [ ]:
show_tables('links')

## 9. Agreement, coactivation, and explanatory examples

For every reference year r and every follow-up year t > r (t = r is covered by the reference units), the D(r)/groups_interventions(t) unit follows each family that was eligible for either idea at selection. On the held-out year-t evaluation patients (returning reference patients and stable entrants counted separately) the frozen rule R and the frozen activation subgroup A_t (medoid activation of the cached year-t embeddings strictly above the frozen cutoff) partition the rows into exactly four disjoint groups: R and A, R and not A, not R and A, and neither. A_t and R are recomputed from the frozen objects and must agree with the Idea 1 and Idea 2 units on the same rows; a difference stops the unit rather than being reconciled. An Idea 2 family that is unavailable inside its (cache-complete) unit is unavailable here with that reason and is left out of the arrays; the Idea 1 yearly rule status does not matter to the groups.

Every group reports rows, patients and deaths (returning and entrant rows separately) and the reliability metrics of section 5 against the constant baseline q fitted on the reference fitting rows inside the same frozen group definition; an empty group is unavailable with its reason and is never zero-filled. The same-year overlap of R and A is Jaccard = |R and A| / |R or A| (a mask overlap inside one year, not an identity overlap across years), with precision, recall, F2 and coverage as in section 6. Original-feature distributions are the mean and population standard deviation (denominator = group rows) of every feature per group. Coactivation reports, for every candidate medoid and every group, prevalence = rows with strictly positive activation / group rows, magnitude = activation sum / group rows and positive magnitude = activation sum / positive rows. The reconstruction error of the medoid's SAE (mean squared scaled-space error) is reported per group so that later sections can compare it with the concept metrics. Post-hoc loss associations are exploratory, and higher activation or predicted risk is not automatically harm.

Examples are deterministic and reference defined: per family and group at most four rows of the group with positive medoid activation, highest activation first, ties by population row order; no label, probability or future loss is read to choose them. For each example the decoder contribution of the medoid factor is removed from the raw embedding after inverse scaling: with scaler u = (h - mean) / scale and tied ReLU decoder rows W, the contribution of factor k is a_k W[k] and the removal in raw coordinates is scale * a_k W[k], so the intervened embedding is h - scale * a_k W[k]. The reconstruction error and the decoder bias are never replaced: zero activation reproduces the embedding exactly, the result equals scale * (x_hat - c_k + e) + mean, and replacing the embedding by its reconstruction or subtracting the scaled contribution without inverse scaling would be different interventions. The activation, the decoder orientation, the explicit decomposition, the zero-removal parity with the cached prediction (rtol 1e-4, atol 1e-5) and the replay of the cached prediction are verified in the unit; a failed gate fails the unit. The intervened embeddings and norm-matched controls (reproducible random directions with exactly the removal's norm, a fixed number of draws per example) pass in one batch through the frozen predictor's serialized head rebuilt from the cached P(r) unit on the run device; the unit reports the probability and Brier-loss change next to the controls, the projection of the removal on the frozen CAVs (kept separate from the predicted change) and the first-order change the cached gradient predicts. These are computational interventions on frozen embeddings; they do not establish clinical causality.


In [ ]:
show_tables('groups', 'coactivation', 'interventions')

## 10. Concurrent and one-year-ahead relationships

This section is an analysis-stage computation over the completed D(r) units (reference performance, Idea 1, Idea 2 frozen view and the four groups); it is not a cached unit, writes nothing to the unit cache and runs in the pilot and analysis profiles on whatever completed. Every table lists its units and marks a missing one `not_run (stopped)`, `unavailable:<reason>` or `failed:<reason>`; nothing is zero-filled, imputed or bridged.

A trajectory is one (reference year r, family, idea): idea 1 follows the activation subgroup A, idea 2 the frozen rule subgroup R. Its loss L[k,t] is the held-out subgroup Brier loss: for r the reference performance unit, for t > r the Idea 1 reliability of A, and for R the exactly pooled Brier of the two groups of the four-group unit that lie inside R (cross-checked against Idea 1 where both exist; a disagreement leaves that year unavailable). delta_loss[t+1] = L[t+1] - L[t] exists only for two actual consecutive years that both have a loss, so a missing year removes the transitions on both sides and is never bridged; a positive change is deterioration and no further threshold is introduced. The forecast origin is the end of year t and labels are assumed complete through t.

Concept metrics, each used alone, are those the units already report: A: activation prevalence, primary-CAV TCAV, original-rule transfer F2, and the medoid SAE's reconstruction error pooled over the A groups; R: medoid retention, medoid prevalence and magnitude inside R, and the reconstruction error pooled over the R groups. The reference-year value exists only where a reference quantity is reported (prevalence, retention, prevalence and magnitude inside R); the other metrics have no value at r, so the first transition simply excludes them.

Calendar boundary. The target year of a transition t -> t+1 is t+1. The common holdout is the final two distinct target years of the configured follow-up grid across all reference experiments; all earlier target years are training. It comes from the configuration, not from which units completed, so an interrupted run cannot move it. For every trajectory and every concept a persistence baseline (delta_loss = 0), an OLS of delta_loss[t+1] on L[t] (history) and an OLS on L[t] and C[t] (history plus one concept), each with an intercept, are scored by absolute error on the held-out transitions only; the models are fitted on transitions whose target year is strictly earlier than every held-out target year, so no holdout-year label or concept enters a fit. History and history-plus-concept use the same transitions (those with the concept at their origin), so the differences are paired. A model needs full column rank and at least one residual degree of freedom; otherwise it is unavailable with its reason and persistence remains. Few transitions per trajectory make most fitted forecasts unavailable; that is reported, not repaired.

Overlap and uncertainty. Experiments of different reference years evaluate the same calendar years and may share patients (one row is one patient-year, so rows shared inside a year are patients shared inside that year); families, ideas and SAE seeds inside one experiment share the same patients. Per target year the table lists the contributing reference years and the shared and total evaluation rows per pair. Counts of trajectories are not independent replicates and no pooled significance is computed. Paired patient-aware resampling needs patient-level probabilities that these aggregate unit payloads do not carry, so uncertainty is reported unavailable; SAE-seed dispersion stays separate and is not a forecast uncertainty. Concurrent associations and the before / concurrent / after / no-deterioration table are descriptive over all consecutive transitions, including the held-out years, and make no forecasting claim; a shared downward time trend is not forecasting evidence. No composite score and no alarm threshold.


In [ ]:
TEMPORAL = pilot.run_temporal_analysis(CACHE, CONFIG)
print("Calendar:", TEMPORAL["calendar"], "; policy:", TEMPORAL["policy"])
print("Forecast uncertainty:", TEMPORAL["summary"]["uncertainty"])
show_tables("temporal_forecast", "temporal_concurrent", "temporal_patterns", "temporal_overlap", "temporal_availability")

## 11. Factor and family summaries

Pending: individual member trajectories before family summaries; overlapping subgroups and SAE seeds are not independent patient replicates. Show support, missingness, ambiguity and negative findings. Interpret only measured outputs.


In [ ]:
show_tables("reference_summary", "temporal_summary")
# Each family trajectory retains medoid/member identity; unavailable years appear as gaps.
import io
import shutil
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(exist_ok=True)
IDENTITIES = {(row["reference_year"], row["id"]): row for row in REPORT["families"] if "id" in row}
for trajectory in TEMPORAL["trajectories"]:
    years = sorted(trajectory["years"])
    values = [np.nan if trajectory["years"][year]["loss"] is None else trajectory["years"][year]["loss"] for year in years]
    identity = IDENTITIES.get((trajectory["reference_year"], trajectory["family_id"]), {})
    fig, axis = plt.subplots(figsize=(6, 3))
    axis.plot(years, values, marker="o")
    axis.set(xlabel="Calendar year", ylabel="Held-out subgroup Brier",
             title=f"r={trajectory['reference_year']} {trajectory['idea']} medoid={identity.get('medoid', 'unknown')}")
    fig.text(.02, .01, f"Family {trajectory['family_id']}; members {identity.get('members', 'unknown')}; missing years = gaps", fontsize=6)
    missing_years = {year: trajectory["years"][year]["reason"] for year in years if trajectory["years"][year]["loss"] is None}
    fig.text(.02, .04, f"Reference {trajectory['reference_year']}; missing {missing_years}; sources listed in completeness.csv", fontsize=5)
    fig.tight_layout(rect=(0, .12, 1, 1))
    identifier = hashlib.sha256(trajectory["key"].encode()).hexdigest()[:16]
    destination = FIGURES / f"trajectory-{identifier}.png"
    image = io.BytesIO()
    fig.savefig(image, format="png", dpi=110)
    image_bytes = image.getvalue()
    assert shutil.disk_usage(OUTPUT).free - len(image_bytes) >= CONFIG["disk_guard_gib"] * 2**30, "Insufficient disk for report figure"
    fd, temporary = tempfile.mkstemp(dir=FIGURES, prefix=".trajectory-")
    with os.fdopen(fd, "wb") as handle:
        handle.write(image_bytes); handle.flush(); os.fsync(handle.fileno())
    os.replace(temporary, destination)
    display(fig)
    plt.close(fig)
    MANIFEST["outputs"].append(str(destination.relative_to(OUTPUT)))
print("Member/seed dispersion remains explanation diagnostics; reference years overlap, never independent replicates.")
print("Interventions remain inspection examples. Population-wide intervention summaries deferred until pilot and analysis end.")

## 12. Exports and sanity checks

Implemented: compact identity-preserving tables, small figures, resolved configuration, manifests, stage evidence and measured time/memory. Undefined numbers become null with reasons. Raw patient data and embeddings remain in the Fiji cache. CUDA prerequisite gates and the R1 pilot remain pending. Compute ends on completion, the disk guard or a user stop; analysis reads completed units only.


In [ ]:
if PROFILE == "v0":
    pilot.run_population_role_checks()
    pilot.run_report_unavailability_checks()
    pilot.run_read_only_cache_checks()
    pilot.run_orchestration_checks()
    pilot.run_unit_cache_checks()
# Export compact aggregates only. Cache retains raw rows, embeddings and activation matrices on Fiji.
import shutil
PROJECTED_EXPORT_BYTES = 3 * len(json.dumps(REPORT, sort_keys=True, allow_nan=False).encode())
assert shutil.disk_usage(OUTPUT).free - PROJECTED_EXPORT_BYTES >= CONFIG["disk_guard_gib"] * 2**30, "Insufficient disk for compact report exports"
MANIFEST["outputs"].extend(pilot.export_report(REPORT, OUTPUT))
COMPLETENESS = REPORT["completeness"]
MISSING = [row for row in COMPLETENESS if row["status"] != "complete"]
SLOWEST = sorted((row for row in COMPLETENESS if row["seconds"] is not None), key=lambda row: -row["seconds"])[:10]
print("Completeness:", len(COMPLETENESS)-len(MISSING), "/", len(COMPLETENESS), "; partial:", bool(MISSING))
print("Slowest units:", SLOWEST)
METRICS = {"complete_units": len(COMPLETENESS)-len(MISSING), "expected_units": len(COMPLETENESS),
           "partial": bool(MISSING), "stop_reason": STOP_REASON, "independent_reference_replicates": False,
           "trajectories": len(TEMPORAL["trajectories"]), "forecast_summary": TEMPORAL["summary"],
           "included_units": INCLUDED_UNITS, "missing_units": MISSING, "slowest_units": SLOWEST}
pilot._atomic_json(OUTPUT / "metrics.json", METRICS)
MANIFEST.update(status="completed", partial=bool(MISSING), stop_reason=STOP_REASON,
                ended_utc=datetime.now(timezone.utc).isoformat(), seconds=time.monotonic()-START_CLOCK,
                projected_export_bytes=PROJECTED_EXPORT_BYTES)
save_progress()
print("Scientific pilot" if PROFILE == "pilot" else PROFILE, "finished; cache", CACHE.directory, "; exports", OUTPUT)